In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
n = 1500

# ====================== Base features ======================
age = np.random.normal(38, 12, n).clip(18, 72).astype(int)

# Strong correlation: higher age → higher income (with noise)
income = (age * 2800 + np.random.normal(0, 18000, n) + 25000).clip(15000, 2500000)
income = np.round(income, -2)

# Spending score correlated with income + some noise + membership effect later
spending_score = (income / 1800 + np.random.normal(0, 18, n)).clip(1, 100).astype(int)

gender = np.random.choice(["Male", "Female", "Other"], n, p=[0.49, 0.48, 0.03])

city = np.random.choice(
    ["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Chennai", "Pune", "Kolkata", "Ahmedabad"],
    n, p=[0.20, 0.16, 0.14, 0.11, 0.11, 0.10, 0.10, 0.08]
)

# Membership strongly related to income + spending
membership_prob = np.clip((income - 30000) / 400000, 0.05, 0.95)
membership = np.where(
    membership_prob > 0.75, "Platinum",
    np.where(membership_prob > 0.55, "Gold",
    np.where(membership_prob > 0.30, "Silver", "Bronze"))
)

# Purchase count correlated with spending_score + membership
purchase_count = (
    spending_score * 0.18
    + np.where(membership == "Platinum", 12,
      np.where(membership == "Gold", 7,
      np.where(membership == "Silver", 3, 0)))
    + np.random.poisson(3, n)
).clip(0, 80).astype(int)

is_active = np.random.choice([True, False], n, p=[0.78, 0.22])

# Last purchase date — more recent for active + high spending customers
days_ago = np.where(
    is_active,
    np.random.exponential(80, n).clip(1, 400),
    np.random.exponential(280, n).clip(60, 700)
).astype(int)

last_purchase = [datetime(2025, 6, 1) - timedelta(days=int(d)) for d in days_ago]

# Feedback related to spending + membership
feedback_map = {
    "Platinum": ["Excellent", "Good"],
    "Gold": ["Excellent", "Good", "Average"],
    "Silver": ["Good", "Average", "Poor"],
    "Bronze": ["Average", "Poor", "Very Poor"]
}
feedback = [
    np.random.choice(feedback_map[m] + [None], p=[0.35, 0.35, 0.2, 0.1][:len(feedback_map[m])+1])
    for m in membership
]

# ====================== Create DataFrame ======================
df = pd.DataFrame({
    "customer_id": range(1001, 1001 + n),
    "age": age,
    "gender": gender,
    "city": city,
    "income": income,
    "spending_score": spending_score,
    "membership": membership,
    "purchase_count": purchase_count,
    "is_active": is_active,
    "last_purchase": last_purchase,
    "feedback": feedback,
})

# ====================== Add nice realistic issues ======================

# 1. Missing values (not completely random)
df.loc[df.sample(60).index, "income"] = np.nan
df.loc[df.sample(35).index, "age"] = np.nan
df.loc[df.query("membership == 'Bronze'").sample(40).index, "feedback"] = None

# 2. Strong outliers
outlier_idx = df.sample(12).index
df.loc[outlier_idx, "income"] = np.random.randint(18_00_000, 45_00_000, 12)

# 3. A few data entry errors / noise
df.loc[df.sample(8).index, "age"] = np.random.randint(95, 120, 8)          # unrealistic age
df.loc[df.sample(6).index, "spending_score"] = np.random.randint(110, 150, 6)  # >100

# 4. High cardinality column (good for profiling)
df["email_domain"] = np.random.choice(
    ["gmail.com", "yahoo.com", "outlook.com", "hotmail.com", "company.in", "edu.in"]
    + [f"domain{i}.com" for i in range(40)],
    n
)

# 5. Derived / interaction feature
df["income_per_purchase"] = (df["income"] / (df["purchase_count"] + 1)).round(0)

df.head(8)
print("\nShape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())


Shape: (1500, 13)

Missing values:
 customer_id              0
age                     35
gender                   0
city                     0
income                  58
spending_score           0
membership               0
purchase_count           0
is_active                0
last_purchase            0
feedback               193
email_domain             0
income_per_purchase     58
dtype: int64


In [7]:
from ydata_profiling import ProfileReport

ModuleNotFoundError: No module named 'ydata_profiling'

In [ ]:
profile = ProfileReport(df, title="Pandas Profiling Report")
profile.to_file("your_report.html")

NameError: name 'ProfileReport' is not defined

In [2]:
df.head()

,customer_id,age,gender,city,income,spending_score,membership,purchase_count,is_active,last_purchase,feedback,email_domain,income_per_purchase
0,1001,43.0,Other,Ahmedabad,159400.0,54,Silver,17,True,2025-05-04,Good,domain35.com,8856.0
1,1002,36.0,Male,Ahmedabad,115900.0,48,Bronze,11,True,2025-03-23,NaN,domain17.com,9658.0
2,1003,45.0,Male,Bangalore,136300.0,68,Bronze,13,False,2024-06-20,Average,domain2.com,9736.0
3,1004,56.0,Male,Mumbai,181700.0,100,Silver,26,True,2025-05-04,Good,domain32.com,6730.0
4,1005,35.0,Female,Hyderabad,119900.0,76,Bronze,18,True,2025-05-27,Average,domain23.com,6311.0
